# Phase 5 — Span-level HITL Simulation

**Plan ref**: §6 Stage 5 (Move 2)

This notebook demonstrates and tests the Phase 5 HITL (Human-In-The-Loop)
correction pipeline:

1. Pre-flight checks (Neo4j, NEEDS_REVIEW page count, schema indexes)
2. Seed PROBLEM_CLASS + schema initialization
3. Simulate span-level corrections on sampled NEEDS_REVIEW pages
4. Write `CORRECTION` nodes + `FEW_SHOT_EXAMPLE` nodes
5. Verify Neo4j writes
6. Write artifact summary to `_artifacts/05_hitl/report.json`

**Idempotency**: Re-running is safe — CORRECTION nodes use content-hashed ids.

In [ ]:
import sys, json, logging
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)

ARTIFACT_DIR = Path('_artifacts/05_hitl')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Smoke-mode: N random NEEDS_REVIEW pages to simulate corrections on
N_SMOKE = 10

print('Setup complete')

In [ ]:
from apps.backend.graph.neo4j_client import get_driver

driver = get_driver()
print('Neo4j connected')

## Pre-flight Checks

In [ ]:
with driver.session() as s:
    total_pages = s.run('MATCH (p:PAGE) RETURN count(p) AS n').single()['n']
    needs_review = s.run(
        "MATCH (p:PAGE) WHERE p.evaluationDecision IN ['needs_review','failed'] "
        "RETURN count(p) AS n"
    ).single()['n']
    corr_count = s.run('MATCH (c:CORRECTION) RETURN count(c) AS n').single()['n']
    fse_count = s.run('MATCH (f:FEW_SHOT_EXAMPLE) RETURN count(f) AS n').single()['n']

print(f'Total pages:        {total_pages:,}')
print(f'Needs review/failed:{needs_review:,}')
print(f'Existing CORRECTION:{corr_count:,}')
print(f'Existing FSE nodes: {fse_count:,}')

assert needs_review >= 0, 'No NEEDS_REVIEW pages — run Phase 4 first'

## Sample NEEDS_REVIEW Pages

In [ ]:
with driver.session() as s:
    sample_pages = s.run(
        "MATCH (p:PAGE) "
        "WHERE p.evaluationDecision IN ['needs_review', 'failed'] "
        "AND p.mode = 'ocr' "
        "RETURN p.id AS page_id, p.textFused AS text, p.problemClass AS cls, "
        "p.interEngineCer AS cer, p.language AS lang, p.tier AS tier "
        "LIMIT $n",
        n=N_SMOKE
    ).data()

print(f'Sampled {len(sample_pages)} pages for simulation')
if sample_pages:
    p = sample_pages[0]
    print(f'  Example: id={p["page_id"][:60]} cls={p["cls"]} cer={p["cer"]} tier={p["tier"]}')

## Simulate Span Corrections

In production, a human reviewer drags a bounding box on the disputed region
and types the corrected text. Here we simulate by taking the first 50 chars
of `textFused` as `before` and appending `[CORRECTED]` as the simulated
correction.

In [ ]:
from apps.backend.feedback.correction_writer import CorrectionInput, write_correction, write_few_shot_example

correction_ids = []

for page in sample_pages:
    text = page.get('text') or ''
    if not text.strip():
        continue

    span = text[:min(50, len(text))]
    corrected = span + '[CORRECTED]'

    inp = CorrectionInput(
        page_id=page['page_id'],
        before=span,
        after=corrected,
        span_bbox=[10.0, 20.0, 200.0, 40.0],   # synthetic bbox
        span_char_range=[0, len(span)],
        problem_class=page.get('cls') or 'OK',
        editor_id='simulation_user',
        human_seconds=12.5,
        language=page.get('lang') or 'zh-classical',
        tier=page.get('tier') or 'primary',
    )
    cid = write_correction(driver, inp)
    correction_ids.append(cid)

    # Simulate a FEW_SHOT_EXAMPLE (image_uri is synthetic for simulation)
    image_uri = f'ancient-pages/{page["page_id"]}/page_0/final.png'
    fse_id = write_few_shot_example(
        driver, cid,
        image_uri=image_uri,
        gt_text=corrected,
        problem_class=page.get('cls') or 'OK',
        language=page.get('lang') or 'zh-classical',
    )

print(f'Wrote {len(correction_ids)} CORRECTION nodes')

## Verify Neo4j Writes

In [ ]:
with driver.session() as s:
    new_corr = s.run('MATCH (c:CORRECTION) RETURN count(c) AS n').single()['n']
    new_fse = s.run('MATCH (f:FEW_SHOT_EXAMPLE) RETURN count(f) AS n').single()['n']
    sample_corr = s.run(
        'MATCH (c:CORRECTION) RETURN c.id, c.pageId, c.problemClass, c.humanSeconds '
        'ORDER BY c.createdAt DESC LIMIT 3'
    ).data()

print(f'CORRECTION count: {new_corr} (+{new_corr - corr_count})')
print(f'FEW_SHOT count:   {new_fse} (+{new_fse - fse_count})')
print('\nLatest 3 CORRECTION nodes:')
for r in sample_corr:
    print(f'  id={r["c.id"]} page={str(r["c.pageId"])[:50]} cls={r["c.problemClass"]}')

## Query Corrections for a Specific Page

In [ ]:
from apps.backend.feedback.correction_writer import get_corrections_for_page

if sample_pages:
    page_id = sample_pages[0]['page_id']
    corrs = get_corrections_for_page(driver, page_id)
    print(f'Corrections for page {page_id[:60]}:')
    for c in corrs:
        print(f'  id={c["id"]} class={c["cls"]} range={c["range"]} secs={c["secs"]}')

## Write Artifact

In [ ]:
import json
from datetime import datetime, timezone

report = {
    'phase': '05_hitl_simulation_span',
    'ts': datetime.now(timezone.utc).isoformat(),
    'pages_sampled': len(sample_pages),
    'corrections_written': len(correction_ids),
    'correction_ids': correction_ids[:5],
    'neo4j_correction_total': new_corr,
    'neo4j_fse_total': new_fse,
}

out = ARTIFACT_DIR / 'report.json'
out.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f'Artifact written → {out}')
print(json.dumps(report, ensure_ascii=False, indent=2))